# 📖 Notebook 3: Aggregations and Faceted Search

Ever used the filter sidebar on Amazon? "Category: Electronics", "Price: $10–$50",
"Rating: 4+ stars"? That's **faceted search** — and it's powered by Elasticsearch
**aggregations**.

## Learning Objectives

By the end of this notebook, you'll understand:
- How to group documents with **bucket aggregations** (like SQL GROUP BY)
- How to compute stats with **metric aggregations** (avg, sum, min, max)
- How to combine search + aggregations for faceted search
- How to nest aggregations for multi-level analysis

## 🛠️ Setup

```bash
cd deep-dives/elasticsearch
docker-compose up -d
```

### Kernel Selection

Select the `.venv` kernel in VS Code's kernel picker (top-right of notebook).  
If it doesn't appear, reload the window: `Cmd+Shift+P` → "Reload Window".

In [ ]:
from elasticsearch import Elasticsearch
from elasticsearch.helpers import bulk
import json

es = Elasticsearch("http://localhost:9200")
info = es.info()
print(f"✅ Connected to Elasticsearch {info['version']['number']}")

## 📦 Setting Up Sample Data

We'll create a products index that resembles an e-commerce store.
This gives us realistic data to aggregate over.

In [ ]:
INDEX_NAME = "products"

if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)

es.indices.create(
    index=INDEX_NAME,
    settings={"number_of_shards": 1, "number_of_replicas": 0},
    mappings={
        "properties": {
            "name": {"type": "text", "fields": {"raw": {"type": "keyword"}}},
            "category": {"type": "keyword"},
            "brand": {"type": "keyword"},
            "price": {"type": "float"},
            "rating": {"type": "float"},
            "reviews_count": {"type": "integer"},
            "in_stock": {"type": "boolean"},
            "tags": {"type": "keyword"},
            "created_at": {"type": "date"}
        }
    }
)

# Sample product data
products = [
    {"name": "Wireless Bluetooth Headphones", "category": "Electronics", "brand": "SoundMax", "price": 49.99, "rating": 4.5, "reviews_count": 1250, "in_stock": True, "tags": ["wireless", "bluetooth", "audio"], "created_at": "2024-01-15"},
    {"name": "Noise Cancelling Earbuds", "category": "Electronics", "brand": "SoundMax", "price": 79.99, "rating": 4.7, "reviews_count": 890, "in_stock": True, "tags": ["wireless", "noise-cancelling", "audio"], "created_at": "2024-03-20"},
    {"name": "USB-C Charging Cable", "category": "Electronics", "brand": "TechBasics", "price": 12.99, "rating": 4.2, "reviews_count": 3400, "in_stock": True, "tags": ["cable", "charging", "usb-c"], "created_at": "2023-11-01"},
    {"name": "Mechanical Keyboard", "category": "Electronics", "brand": "KeyMaster", "price": 129.99, "rating": 4.8, "reviews_count": 670, "in_stock": False, "tags": ["keyboard", "mechanical", "gaming"], "created_at": "2024-02-10"},
    {"name": "Running Shoes Pro", "category": "Sports", "brand": "SpeedFit", "price": 89.99, "rating": 4.6, "reviews_count": 2100, "in_stock": True, "tags": ["running", "shoes", "athletic"], "created_at": "2024-01-05"},
    {"name": "Yoga Mat Premium", "category": "Sports", "brand": "FlexZone", "price": 34.99, "rating": 4.4, "reviews_count": 1800, "in_stock": True, "tags": ["yoga", "fitness", "mat"], "created_at": "2023-12-15"},
    {"name": "Resistance Bands Set", "category": "Sports", "brand": "FlexZone", "price": 19.99, "rating": 4.3, "reviews_count": 950, "in_stock": True, "tags": ["fitness", "resistance", "workout"], "created_at": "2024-02-28"},
    {"name": "Water Bottle Insulated", "category": "Sports", "brand": "HydroLife", "price": 24.99, "rating": 4.5, "reviews_count": 4200, "in_stock": True, "tags": ["bottle", "hydration", "insulated"], "created_at": "2023-10-10"},
    {"name": "Python Programming Book", "category": "Books", "brand": "TechPress", "price": 39.99, "rating": 4.6, "reviews_count": 520, "in_stock": True, "tags": ["programming", "python", "education"], "created_at": "2024-01-20"},
    {"name": "System Design Interview Guide", "category": "Books", "brand": "TechPress", "price": 44.99, "rating": 4.9, "reviews_count": 310, "in_stock": False, "tags": ["system-design", "interview", "education"], "created_at": "2024-03-01"},
    {"name": "Science Fiction Collection", "category": "Books", "brand": "ReadMore", "price": 15.99, "rating": 4.1, "reviews_count": 780, "in_stock": True, "tags": ["fiction", "sci-fi", "collection"], "created_at": "2023-09-15"},
    {"name": "Coffee Maker Deluxe", "category": "Home", "brand": "BrewPerfect", "price": 69.99, "rating": 4.3, "reviews_count": 1500, "in_stock": True, "tags": ["coffee", "kitchen", "appliance"], "created_at": "2024-01-10"},
    {"name": "LED Desk Lamp", "category": "Home", "brand": "BrightLife", "price": 29.99, "rating": 4.4, "reviews_count": 2200, "in_stock": True, "tags": ["lamp", "led", "desk"], "created_at": "2023-11-20"},
    {"name": "Air Purifier Compact", "category": "Home", "brand": "CleanAir", "price": 149.99, "rating": 4.7, "reviews_count": 430, "in_stock": True, "tags": ["air-purifier", "health", "home"], "created_at": "2024-02-05"},
    {"name": "Stainless Steel Cookware Set", "category": "Home", "brand": "ChefPro", "price": 199.99, "rating": 4.8, "reviews_count": 360, "in_stock": True, "tags": ["cookware", "kitchen", "steel"], "created_at": "2024-03-10"},
]

actions = [{"_index": INDEX_NAME, "_id": i+1, "_source": p} for i, p in enumerate(products)]
success, _ = bulk(es, actions)
es.indices.refresh(index=INDEX_NAME)

print(f"✅ Indexed {success} products into '{INDEX_NAME}'")

## 🪣 Bucket Aggregations (GROUP BY)

Bucket aggregations group documents into "buckets". Each bucket contains
documents that share something in common.

Think of it like SQL's `GROUP BY`:

```sql
SELECT category, COUNT(*) FROM products GROUP BY category
```

The Elasticsearch equivalent is a **terms** aggregation.

In [ ]:
# Terms aggregation: count products per category
print("🪣 Products per Category")
print("=" * 40)

results = es.search(
    index=INDEX_NAME,
    size=0,  # we only want aggregations, not search results
    aggs={
        "categories": {
            "terms": {
                "field": "category"
            }
        }
    }
)

for bucket in results["aggregations"]["categories"]["buckets"]:
    bar = "█" * bucket["doc_count"]
    print(f"  {bucket['key']:<15} {bucket['doc_count']} products  {bar}")

print("\n💡 'size=0' means we don't want search hits — just the aggregation results.")
print("   This is faster because Elasticsearch skips fetching document _source.")

In [ ]:
# Terms aggregation on tags (each product can have multiple tags)
print("🏷️ Top 10 Most Common Tags")
print("=" * 40)

results = es.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "popular_tags": {
            "terms": {
                "field": "tags",
                "size": 10  # top 10 tags
            }
        }
    }
)

for bucket in results["aggregations"]["popular_tags"]["buckets"]:
    bar = "█" * bucket["doc_count"]
    print(f"  {bucket['key']:<20} {bucket['doc_count']}  {bar}")

### Range Aggregation: Price Buckets

Instead of grouping by exact values, you can create custom ranges.
This is how e-commerce sites show price range filters.

In [ ]:
# Range aggregation: price ranges
print("💰 Products by Price Range")
print("=" * 40)

results = es.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "price_ranges": {
            "range": {
                "field": "price",
                "ranges": [
                    {"key": "Budget (under $25)", "to": 25},
                    {"key": "Mid-range ($25-$75)", "from": 25, "to": 75},
                    {"key": "Premium ($75-$150)", "from": 75, "to": 150},
                    {"key": "Luxury ($150+)", "from": 150}
                ]
            }
        }
    }
)

for bucket in results["aggregations"]["price_ranges"]["buckets"]:
    bar = "█" * bucket["doc_count"]
    print(f"  {bucket['key']:<25} {bucket['doc_count']} products  {bar}")

## 📊 Metric Aggregations (AVG, SUM, MIN, MAX)

Metric aggregations compute statistics across your documents.
Think of them like SQL aggregate functions:

```sql
SELECT AVG(price), MIN(price), MAX(price) FROM products
```

In [ ]:
# Multiple metric aggregations at once
print("📊 Product Price Statistics")
print("=" * 40)

results = es.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "avg_price": {"avg": {"field": "price"}},
        "min_price": {"min": {"field": "price"}},
        "max_price": {"max": {"field": "price"}},
        "total_revenue": {"sum": {"field": "price"}},
        "price_stats": {"stats": {"field": "price"}},  # all stats in one!
        "avg_rating": {"avg": {"field": "rating"}},
        "total_reviews": {"sum": {"field": "reviews_count"}}
    }
)

aggs = results["aggregations"]
print(f"  Average price:    ${aggs['avg_price']['value']:.2f}")
print(f"  Cheapest:         ${aggs['min_price']['value']:.2f}")
print(f"  Most expensive:   ${aggs['max_price']['value']:.2f}")
print(f"  Total value:      ${aggs['total_revenue']['value']:.2f}")
print(f"  Average rating:   {aggs['avg_rating']['value']:.1f} ⭐")
print(f"  Total reviews:    {int(aggs['total_reviews']['value']):,}")

print("\n💡 The 'stats' aggregation returns count, min, max, avg, and sum in one call!")

## 🔗 Nested Aggregations: Buckets + Metrics

The real power comes from **nesting** metric aggregations inside bucket
aggregations. This lets you compute stats for each group:

```sql
SELECT category, AVG(price), AVG(rating)
FROM products
GROUP BY category
```

In [ ]:
# Nested: average price and rating per category
print("📊 Stats per Category")
print("=" * 65)

results = es.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "by_category": {
            "terms": {"field": "category"},
            "aggs": {
                "avg_price": {"avg": {"field": "price"}},
                "avg_rating": {"avg": {"field": "rating"}},
                "total_reviews": {"sum": {"field": "reviews_count"}}
            }
        }
    }
)

print(f"  {'Category':<15} {'Avg Price':>10} {'Avg Rating':>12} {'Total Reviews':>15}")
print("  " + "-" * 55)

for bucket in results["aggregations"]["by_category"]["buckets"]:
    name = bucket["key"]
    avg_p = bucket["avg_price"]["value"]
    avg_r = bucket["avg_rating"]["value"]
    reviews = int(bucket["total_reviews"]["value"])
    print(f"  {name:<15} ${avg_p:>8.2f} {avg_r:>10.1f} ⭐ {reviews:>13,}")

In [ ]:
# Multi-level nesting: brands within categories
print("🏢 Brands within each Category")
print("=" * 50)

results = es.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "by_category": {
            "terms": {"field": "category"},
            "aggs": {
                "by_brand": {
                    "terms": {"field": "brand"},
                    "aggs": {
                        "avg_price": {"avg": {"field": "price"}}
                    }
                }
            }
        }
    }
)

for cat_bucket in results["aggregations"]["by_category"]["buckets"]:
    print(f"\n  📁 {cat_bucket['key']} ({cat_bucket['doc_count']} products)")
    for brand_bucket in cat_bucket["by_brand"]["buckets"]:
        avg_p = brand_bucket["avg_price"]["value"]
        print(f"     └── {brand_bucket['key']:<15} {brand_bucket['doc_count']} product(s), avg ${avg_p:.2f}")

## 🛒 Faceted Search: Search + Aggregations Together

Faceted search is the killer feature for e-commerce. It combines:
1. A **search query** to find matching products
2. **Aggregations** to build the filter sidebar

When a user searches for "wireless", we want to show:
- The matching products
- Filter counts: how many results in each category, price range, brand, etc.

This happens in a **single Elasticsearch request**!

In [ ]:
def faceted_search(query_text, filters=None):
    """Perform a faceted search like an e-commerce site."""
    # Build the query
    must_clauses = [{"multi_match": {"query": query_text, "fields": ["name", "tags"]}}]
    filter_clauses = []

    if filters:
        if "category" in filters:
            filter_clauses.append({"term": {"category": filters["category"]}})
        if "max_price" in filters:
            filter_clauses.append({"range": {"price": {"lte": filters["max_price"]}}})
        if "in_stock" in filters:
            filter_clauses.append({"term": {"in_stock": filters["in_stock"]}})

    body = {
        "query": {
            "bool": {
                "must": must_clauses,
                "filter": filter_clauses
            }
        },
        "aggs": {
            "categories": {"terms": {"field": "category"}},
            "brands": {"terms": {"field": "brand"}},
            "price_ranges": {
                "range": {
                    "field": "price",
                    "ranges": [
                        {"key": "Under $25", "to": 25},
                        {"key": "$25 – $75", "from": 25, "to": 75},
                        {"key": "$75+", "from": 75}
                    ]
                }
            },
            "avg_price": {"avg": {"field": "price"}},
            "in_stock_count": {
                "filter": {"term": {"in_stock": True}},
                "aggs": {"count": {"value_count": {"field": "in_stock"}}}
            }
        },
        "size": 10
    }

    return es.search(index=INDEX_NAME, body=body)


def display_faceted_results(results, query):
    """Display results like an e-commerce page."""
    hits = results["hits"]["hits"]
    total = results["hits"]["total"]["value"]
    aggs = results["aggregations"]

    print(f"\n🔍 Search: \"{query}\" — {total} result(s)")
    print("=" * 60)

    # Sidebar: facets
    print("\n┌─── 📋 FILTER SIDEBAR ─────────────────────┐")
    print("│                                            │")

    print("│  Category:                                 │")
    for b in aggs["categories"]["buckets"]:
        print(f"│    ☐ {b['key']:<20} ({b['doc_count']})          │"[:46] + "│")

    print("│                                            │")
    print("│  Price Range:                              │")
    for b in aggs["price_ranges"]["buckets"]:
        print(f"│    ☐ {b['key']:<20} ({b['doc_count']})          │"[:46] + "│")

    print("│                                            │")
    print("│  Brand:                                    │")
    for b in aggs["brands"]["buckets"]:
        print(f"│    ☐ {b['key']:<20} ({b['doc_count']})          │"[:46] + "│")

    in_stock = aggs["in_stock_count"]["count"]["value"]
    print("│                                            │")
    print(f"│  ☐ In Stock Only ({int(in_stock)})                    │"[:46] + "│")
    print("│                                            │")
    print("└────────────────────────────────────────────┘")

    # Main content: search results
    print(f"\n📦 Results (avg price: ${aggs['avg_price']['value']:.2f}):")
    for hit in hits:
        src = hit["_source"]
        stock = "✅" if src["in_stock"] else "❌"
        print(f"  {stock} ${src['price']:<8.2f} {src['name']:<40} [{src['category']}] {src['rating']}⭐")

    return results

In [ ]:
# Faceted search: no filters
results = faceted_search("fitness")
display_faceted_results(results, "fitness");

In [ ]:
# Faceted search: with a category filter applied
results = faceted_search("kitchen", filters={"category": "Home", "in_stock": True})
display_faceted_results(results, "kitchen (Home, in stock only)");

## 🔢 Cardinality & Percentiles: Two More Must-Know Metrics

Two aggregations come up constantly in real analytics dashboards:

| Aggregation | Answers | Example question |
|-------------|---------|------------------|
| `cardinality` | "How many **unique** values?" | *How many distinct brands do we sell?* |
| `percentiles` | "What's the value at the Nth percentile?" | *What price is the 95th percentile of our catalog?* |

`cardinality` uses the [HyperLogLog++](https://en.wikipedia.org/wiki/HyperLogLog)
algorithm under the hood. That means it's an **approximation** (±1–5% error by
default), but it uses bounded memory — so it scales to billions of distinct
values without blowing up your heap. For most dashboards, this is exactly
what you want.


In [ ]:
# Cardinality: how many unique brands and categories do we have?
# Percentiles: what's the typical price distribution?
print("🔢 Cardinality + Percentiles")
print("=" * 45)

r = es.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "unique_brands":     {"cardinality": {"field": "brand"}},
        "unique_categories": {"cardinality": {"field": "category"}},
        "unique_tags":       {"cardinality": {"field": "tags"}},
        "price_percentiles": {
            "percentiles": {
                "field": "price",
                "percents": [25, 50, 75, 95, 99]
            }
        }
    }
)

a = r["aggregations"]
print(f"  Unique brands:      {a['unique_brands']['value']}")
print(f"  Unique categories:  {a['unique_categories']['value']}")
print(f"  Unique tags:        {a['unique_tags']['value']}")
print()
print("  Price percentiles:")
for p, v in a["price_percentiles"]["values"].items():
    print(f"    p{p:<5}  ${v:.2f}")

print("\n💡 Cardinality is approximate (HyperLogLog++) — trades exactness for")
print("   constant memory. Perfect for 'unique visitors' style metrics.")


## 📈 Histogram Aggregation

Histograms create evenly-spaced buckets — perfect for price distribution charts.

In [ ]:
# Histogram: price distribution in $25 increments
print("📈 Price Distribution (histogram, $25 buckets)")
print("=" * 50)

results = es.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "price_histogram": {
            "histogram": {
                "field": "price",
                "interval": 25
            }
        }
    }
)

for bucket in results["aggregations"]["price_histogram"]["buckets"]:
    low = bucket["key"]
    count = bucket["doc_count"]
    bar = "█" * count
    print(f"  ${low:>6.0f} – ${low+25:<6.0f}  {count}  {bar}")

## 📅 Date Histogram

Group documents by time intervals — great for showing trends over time.

In [ ]:
# Date histogram: products added per month
print("📅 Products Added per Month")
print("=" * 40)

results = es.search(
    index=INDEX_NAME,
    size=0,
    aggs={
        "by_month": {
            "date_histogram": {
                "field": "created_at",
                "calendar_interval": "month",
                "format": "yyyy-MM"
            }
        }
    }
)

for bucket in results["aggregations"]["by_month"]["buckets"]:
    count = bucket["doc_count"]
    bar = "█" * count
    print(f"  {bucket['key_as_string']}  {count} product(s)  {bar}")

## 🧪 Exercises

1. **Top-rated brands**: Write an aggregation to find the brand with the highest average rating
2. **Stock analysis**: How many products are in stock vs out of stock per category?
3. **Price percentiles**: Use a `percentiles` aggregation on the price field to find the 25th, 50th, 75th, and 99th percentiles
4. **Custom faceted search**: Modify the `faceted_search` function to add a "rating" facet (e.g., 4+ stars, 3+ stars)

In [ ]:
# Exercise space — try your aggregations here!


## 🎯 Key Takeaways

1. **Bucket aggregations** group documents (like `GROUP BY`): `terms`, `range`, `histogram`, `date_histogram`
2. **Metric aggregations** compute stats: `avg`, `sum`, `min`, `max`, `stats`, `percentiles`
3. **Nested aggregations** = metrics inside buckets → stats per group
4. **Faceted search** = search + aggregations in one request → filter sidebar + results
5. Use `size=0` when you only need aggregations (no search results) for better performance
6. Aggregations work on `keyword` and numeric fields — not on `text` fields

**Next up**: Notebook 4 covers **Scaling Elasticsearch Clusters** — shards,
replicas, node types, and how Elasticsearch works under the hood.

## 🧹 Cleanup

In [ ]:
if es.indices.exists(index=INDEX_NAME):
    es.indices.delete(index=INDEX_NAME)
    print(f"🗑️  Deleted '{INDEX_NAME}'")
print("✅ Cleanup complete")